# H1 · Has the "agility" of the top 20 scorers increased?

## The question

> It is claimed that the average agility of the players in the top 20 of each
> season has increased compared to the past. Agility can be defined as the
> ratio of a player's height to weight. To test this claim, compare the data of
> seasons 2022-23 through 2023-24 with the data of 2020-21 through 2021-22.

## Reading the question

**"Agility."** The claim brings its own definition: height divided by weight.
That is the number tested below, and it is worth saying plainly what it
measures. Trae Young is 188 cm and 74.4 kg, so he scores 2.53. Nikola Jokić is
210.8 cm and 128.8 kg, so he scores 1.64. The ratio says Young carries less mass
per centimetre of frame. It says nothing about how fast either man changes
direction, how quickly he leaves the floor, or whether he can stay in front of a
defender, which is what a coach means by agility. This is a build measurement
borrowing the word, and no result below should be read as being about
athleticism.

A second property of the metric decides how the rest of the notebook has to be
read. **Height and weight come from a player's bio page, and each player has
exactly one pair of them.** Across all 4,466 player-seasons in the database, not
one player's ratio changes from one season to the next. Jokić is 210.8 cm and
128.8 kg in 2020-21 and identical in 2023-24. The metric cannot see a player
getting leaner, because within a player it never moves. A group average can
change only if the top 20 is made of different people. So the claim is about
players becoming more agile, and the data can answer only whether the leading
scorers are now a leaner-framed set of men.

**"The top 20 of each season."** The twenty highest point scorers, ranks 1 to
20. This data has no wins column and no all-round rating, so "top 20" cannot
mean the twenty best players. It is the order Basketball-Reference sorts its own
season pages in. Everything below is about scoring volume.

**The two windows.** Seasons are stored under their ending year, so the recent
group is 2023 and 2024 and the earlier group is 2021 and 2022. Twenty players a
season, two seasons a group: 40 player-seasons on each side, exactly. The
database now runs through 2025-26, but the claim names its own seasons and the
window stays where the claim put it.

**What is probably doing the work.** The ratio tracks position closely. Across
these 80 player-seasons it averages 2.20 at point guard and 1.71 at centre, a
gap of 0.49 against a standard deviation of 0.22 for the whole set. A group mean
moves whenever the position mix moves, and a position mix has nothing to do with
anyone's fitness. If the number shifts, the first question is which positions
arrived and which left. That check is below.

## The hypothesis, fixed before the data is touched

The claim points one way: agility has increased. Written out,

- **H₀** the mean height-to-weight ratio of the top 20 in 2022-23 and 2023-24 is
  no higher than it was in 2020-21 and 2021-22.
- **H₁** it is higher.
- **α = 0.05.**
- **One-tailed, upper tail.** The claim is directional, so a one-tailed test
  spends all of its power on the side the claim lives on.

One-tailed testing has a failure mode this project has already walked into. The
companion H2 analysis ran one-tailed, produced a t-statistic of −4.9, and
reported "cannot reject H₀". Both halves of that sentence were correct.
Together they buried a large effect running the other way, because a one-sided
frame leaves the opposite direction nowhere to be reported.

Two rules, fixed here rather than after the numbers are in.

1. Read the sign of the difference before the p-value. A one-tailed
   non-rejection means there is no evidence of an *increase*. It never means the
   two groups are alike.
2. Print the two-tailed p-value next to it. That costs one number and removes
   the ambiguity entirely.

The recent period is the first category of the grouping variable, which makes it
group A in `compare_groups`. A positive estimate then reads "recent above
earlier", and `alternative='greater'` reads as the claim. The sign is checked
against the two group means before anything is interpreted.

## What we need, and where it comes from

For each of the four seasons from 2020-21 to 2023-24, the twenty players who
scored the most points that year. For each of those players: how tall he is,
what he weighs, and which position he played that season.

It is one lookup. The database keeps a table with one row per player per season,
and that row already carries the season's scoring rank beside the player's
height, weight and the ratio of the two. The ratio was worked out when the data
was loaded rather than here, so this notebook cannot divide the two numbers
differently from any other. Heights and weights were converted to centimetres
and kilograms at the same time, because the source publishes feet, inches and
pounds.

In [1]:
import _setup  # noqa: F401

import pandas as pd

from utils.custom_plots import cross_tab_heatmap, ecdf_plot, grouped_box_plot
from utils.custom_stats import (
    compare_groups,
    normality_test,
    summary_stats,
    variance_test,
)
from utils.db_utils import run_query

In [2]:
SQL = """
SELECT season,
       season_label,
       player_id,
       player_name,
       position,
       points_rank,
       height_cm,
       weight_kg,
       height_to_weight
FROM analyst_ready.player_season
WHERE points_rank <= 20
  AND season BETWEEN 2021 AND 2024
ORDER BY season, points_rank
"""

top20 = run_query(SQL)

# SQL `numeric` arrives as Decimal on some drivers, and the plotting helpers
# skip object columns without saying so. Cast once, here.
for col in ("height_cm", "weight_kg", "height_to_weight"):
    top20[col] = top20[col].astype(float)

print(top20.shape)
top20.head()

(80, 9)


,season,season_label,player_id,player_name,position,points_rank,height_cm,weight_kg,height_to_weight
0,2021,2020-21,curryst01,Stephen Curry,PG,1,188.0,83.9,2.2408
1,2021,2020-21,lillada01,Damian Lillard,PG,2,188.0,90.7,2.0728
2,2021,2020-21,jokicni01,Nikola Jokić,C,3,210.8,128.8,1.6366
3,2021,2020-21,bealbr01,Bradley Beal,SG,4,193.0,93.9,2.0554
4,2021,2020-21,doncilu01,Luka Dončić,PG,5,203.2,104.3,1.9482


In [3]:
RECENT = "recent · 2022-23 & 2023-24"
EARLIER = "earlier · 2020-21 & 2021-22"

# Recent is the first category, so compare_groups treats it as group A. A
# positive estimate then means "recent above earlier", which is the claim.
top20["period"] = pd.Categorical(
    ["recent" if s >= 2023 else "earlier" for s in top20["season"]],
    categories=["recent", "earlier"],
    ordered=True,
).rename_categories([RECENT, EARLIER])

# Positions ordered shortest to tallest so every chart reads left to right.
POSITIONS = ["PG", "SG", "SF", "PF", "C"]
top20["position"] = pd.Categorical(
    top20["position"], categories=POSITIONS, ordered=True
)

print(
    top20.groupby("period", observed=True).agg(
        player_seasons=("player_id", "size"),
        distinct_players=("player_id", "nunique"),
    ),
    end="\n\n",
)

print("distinct height values :", top20["height_cm"].nunique())
print("distinct weight values :", top20["weight_kg"].nunique())
print("distinct ratio values  :", top20["height_to_weight"].nunique())
print("distinct players       :", top20["player_id"].nunique())

                             player_seasons  distinct_players
period                                                       
recent · 2022-23 & 2023-24               40                30
earlier · 2020-21 & 2021-22              40                30

distinct height values : 12
distinct weight values : 27
distinct ratio values  : 39
distinct players       : 41


Forty player-seasons a side, as the design guarantees. They come from 30 players
in each period and 41 across both, which already says the two groups are not
built from separate people. That gets its own section further down.

The distinct-value counts matter for how precisely any of this can be quoted.
Eighty rows carry only 12 different heights, because Basketball-Reference
publishes height in whole inches and 2.54 cm is the smallest step the figure can
take. Weight is recorded to the pound and lands on 27 values. One rung of the
height grid moves the ratio by about 0.027 for a typical 200 cm, 95 kg player,
so the second decimal place is the last one the measurement actually supports,
whatever the tables below print. The ratio ends up with 39 distinct values
across 41 players, which is close to one per man: within this group it is
almost an identifier.

## The two distributions

An empirical cumulative distribution puts both groups on one axis with no
bin-width to choose. Each curve climbs from 0 to 1 as you move right, and its
height at any point reads as "this share of the group is at or below this
ratio".

In [4]:
ecdf_plot(
    top20,
    cols="height_to_weight",
    group_col="period",
    mark_percentiles=[0.25, 0.5, 0.75],
    title="Height ÷ weight for the season's top 20 scorers, recent vs earlier",
)

The two curves sit on top of each other. All three quartile markers land almost
in the same place: lower quartiles at 1.890 and 1.895, medians at 1.985 and
1.975, upper quartiles at 2.163 and 2.157. The largest of those three gaps is
0.010, roughly a third of one rung of the height grid.

The only visible daylight is in the tails, and each side is one player. The
earlier curve reaches further left because Zion Williamson is in it: 198.1 cm
and 128.8 kg gives him 1.538, the heaviest frame for its height anywhere in the
80 rows. Above 2.25 the recent curve climbs slightly faster, which is the same
effect from the other end. Both groups stop at the identical 2.527, Trae Young,
who appears in three of these four seasons.

Whatever the test reports, it is describing something that does not show up on
this chart.

In [5]:
ratio_summary = pd.concat(
    [
        summary_stats(sub, cols=["height_to_weight"]).assign(period=name)
        for name, sub in top20.groupby("period", observed=True)
    ],
    ignore_index=True,
)

ratio_summary[
    ["period", "n", "mean", "ci_low", "ci_high", "median", "std", "iqr",
     "skew", "min", "max"]
].round(4)

,period,n,mean,ci_low,ci_high,median,std,iqr,skew,min,max
0,recent · 2022-23 & 2023-24,40,2.0210,1.9568,2.0851,1.9850,0.2006,0.2724,0.1917,1.6366,2.5269
1,earlier · 2020-21 & 2021-22,40,2.0244,1.9468,2.1019,1.9754,0.2426,0.2613,0.1876,1.5380,2.5269


The recent mean is 2.0210 and the earlier mean is 2.0244. The recent group is
lower by 0.0034, which is the wrong direction for the claim and a size not worth
the word "direction". It is one eighth of a single rung of the height grid, and
1.5 % of one standard deviation. Two players swapping between the groups would
move it further than that.

The confidence intervals on the two means overlap almost completely, 1.957 to
2.085 against 1.947 to 2.102. Each interval is around forty times as wide as the
gap between the two point estimates.

The spread columns behave the same way: standard deviations of 0.201 and 0.243,
interquartile ranges of 0.272 and 0.261. The earlier group is slightly wider,
and Zion Williamson is about a third of the reason. Take his one row out and the
standard deviations are 0.201 and 0.232. Neither group is skewed enough to
worry a mean.

## Which test, and why

`compare_groups(test='auto')` screens normality and equal variance itself and
records the reasoning. Running the two screens separately first makes it
possible to read that reasoning rather than take it on trust.

In [6]:
shape_checks = pd.concat(
    [
        normality_test(sub, cols=["height_to_weight"]).assign(period=name)
        for name, sub in top20.groupby("period", observed=True)
    ],
    ignore_index=True,
)

print(
    shape_checks[
        ["period", "n", "statistic", "p_value", "decision", "skew",
         "excess_kurtosis", "flags"]
    ].round(4).to_string(index=False),
    end="\n\n",
)

variance_test(top20, group_col="period", value_col="height_to_weight")[
    ["test", "group_sizes", "statistic", "p_value", "decision", "max_var_ratio"]
].round(4)

                     period  n  statistic  p_value          decision   skew  excess_kurtosis flags
 recent · 2022-23 & 2023-24 40     0.9818   0.7555 fail to reject H₀ 0.1917          -0.1238      
earlier · 2020-21 & 2021-22 40     0.9793   0.6628 fail to reject H₀ 0.1876          -0.2708      



,test,group_sizes,statistic,p_value,decision,max_var_ratio
0,levene(median),recent · 2022-23 & 2023-24=40; earlier · 2020-...,1.0268,0.3141,fail to reject H₀,1.463


Both groups pass Shapiro-Wilk comfortably, p = 0.76 and p = 0.66, with skew near
0.19 and slightly light tails on both sides. Levene gives p = 0.31 and a
variance ratio of 1.46, so the spreads are close enough to treat as equal.

Neither result should be over-read at n = 40. A normality test on forty
observations has little power, so passing it means "nothing obviously wrong"
rather than "normal". What matters is that the ECDF shows the same thing, and
that a t-test on the mean of forty values tolerates mild departures anyway. The
route to Welch's t-test is a reasonable one, and Welch stays correct even if the
equal-variance finding is wrong.

The coarse-grid problem that dogs height on its own is much weaker here. Height
alone takes 12 values in these 80 rows, few enough that a normality test would
reject it for being discrete rather than for being misshapen. Dividing by a
weight recorded to the pound spreads those 12 onto 39, which is enough to stop
the grid driving the answer.

In [7]:
shared_args = dict(
    df=top20, group_col="period", value_col="height_to_weight",
    test="auto", ci="bootstrap",
)

result = pd.concat(
    [
        compare_groups(**shared_args, alternative="greater"),
        compare_groups(**shared_args, alternative="two-sided"),
    ],
    ignore_index=True,
)

# The sign only reads correctly if group A really is the recent period.
print("group A is:", result.loc[0, "comparison"].split(" vs ")[0])
print("mean A - mean B (direct):",
      round(top20.groupby("period", observed=True)["height_to_weight"]
            .mean().pipe(lambda m: m[RECENT] - m[EARLIER]), 4))

result.T

group A is: recent · 2022-23 & 2023-24
mean A - mean B (direct): -0.0034


,0,1
comparison,recent · 2022-23 & 2023-24 vs earlier · 2020-2...,recent · 2022-23 & 2023-24 vs earlier · 2020-2...
n_groups,2,2
group_sizes,recent · 2022-23 & 2023-24=40; earlier · 2020-...,recent · 2022-23 & 2023-24=40; earlier · 2020-...
test,welch,welch
n,80,80
statistic,-0.068163,-0.068163
p_value,0.527082,0.945837
decision,fail to reject H₀,fail to reject H₀
estimate_type,mean difference,mean difference
estimate,-0.003392,-0.003392


### Reading the result, direction first

`chosen_because` says `auto: normal 2/2, variances equal`, so both groups passed
the normality screen and the routine went to Welch's t-test on the means. That
matches the two checks above. `flags` came back empty in both rows, which means
the routine found nothing it wanted to warn about, including nothing about group
size.

**The direction.** The estimate is −0.0034, and the group means confirm it
independently: 2.0210 recent against 2.0244 earlier. The recent top 20 is
marginally *less* lean, not more. So the claim does not merely fail to be
supported, it points the wrong way, and that has to be said before the p-value
rather than after.

**The size.** Hedges' g is −0.015, which the toolkit labels negligible and which
is negligible on any reading. For scale, the average point guard in this data
sits 0.13 above the average shooting guard, and the average power forward sits
0.18 above the average centre. Those are the sorts of gaps this metric was built
to show. This one is one seventieth of a standard deviation.

**The p-values.**

| | p |
| --- | ---: |
| One-tailed, upper (the pre-registered test) | 0.527 |
| Two-tailed | 0.946 |

Both fail to reject H₀, and here the two readings genuinely agree. There is no
evidence of an increase, and none of a decrease either. The two-tailed number is
what makes the second half of that sentence sayable, which is the whole reason
it is printed.

**The interval.** The bootstrapped 95 % interval on the mean difference runs
from −0.105 to +0.090. That is the useful line in the table, and it is a
statement about power rather than about basketball. It says the study could rule
out only a shift larger than about 0.1, which is roughly a fifth of the distance
from an average point guard to an average centre. A real change smaller than
that would have passed through this test unseen. Forty player-seasons a side is
not much, and the next section shows they are worth even less than forty.

## The same men are in both groups

A two-sample t-test assumes the two samples are independent. These are not. The
leading scorers do not turn over much from one two-year window to the next, and
because the ratio is a fixed bio figure, a player who appears in both periods
contributes the *identical* number to each side.

That is worse than ordinary non-independence. Stephen Curry scores 2.2408 in the
earlier group and 2.2408 in the recent one, so he cannot express any difference
between the two windows at all. Every row like his pulls the estimated gap
toward zero while still counting toward n and shrinking the standard error. The
test comes out diluted and overconfident at the same time.

Since the value is constant per player, the correction is exact rather than
approximate: collapse each period to one row per player and run the same
comparison.

In [8]:
seen = {name: set(sub["player_id"])
        for name, sub in top20.groupby("period", observed=True)}
shared = seen[RECENT] & seen[EARLIER]

print(f"{len(shared)} of {top20['player_id'].nunique()} players are in both periods")
for name in (RECENT, EARLIER):
    sub = top20[top20["period"] == name]
    n_shared = int(sub["player_id"].isin(shared).sum())
    print(f"  {name}: {n_shared} of {len(sub)} rows ({n_shared / len(sub):.0%})")

print("\nplayers whose ratio differs between their own seasons:",
      int((top20.groupby("player_id")["height_to_weight"].nunique() > 1).sum()))

print("\nin both periods:")
print(", ".join(sorted(top20.loc[top20["player_id"].isin(shared),
                                 "player_name"].unique())))

# One row per player per period, so nobody is counted twice inside a group.
per_player = top20.drop_duplicates(subset=["period", "player_id"])
compare_groups(
    per_player, group_col="period", value_col="height_to_weight",
    test="auto", alternative="two-sided",
).T

19 of 41 players are in both periods
  recent · 2022-23 & 2023-24: 28 of 40 rows (70%)
  earlier · 2020-21 & 2021-22: 29 of 40 rows (72%)

players whose ratio differs between their own seasons: 0

in both periods:
Anthony Edwards, Damian Lillard, De'Aaron Fox, DeMar DeRozan, Devin Booker, Donovan Mitchell, Giannis Antetokounmpo, Jaylen Brown, Jayson Tatum, Joel Embiid, Julius Randle, Kevin Durant, LeBron James, Luka Dončić, Nikola Jokić, Pascal Siakam, Stephen Curry, Trae Young, Zach LaVine


,0
comparison,recent · 2022-23 & 2023-24 vs earlier · 2020-2...
n_groups,2
group_sizes,recent · 2022-23 & 2023-24=30; earlier · 2020-...
test,welch
n,60
statistic,0.135577
p_value,0.892636
decision,fail to reject H₀
estimate_type,mean difference
estimate,0.007817


19 of the 41 players are in both periods, and they supply 28 of the 40 recent
rows and 29 of the 40 earlier rows. Roughly 70 % of each group is made of the
same men, carrying the same number on both sides. Curry, Jokić, Dončić,
Antetokounmpo, Durant, Embiid, Tatum and twelve others sit in this data twice
with nothing to say about the difference between the two windows.

Collapsing to one row per player leaves 30 a side, and the comparison changes in
the one way that matters: **the sign flips.** The estimate goes from −0.0034 to
+0.0078, so the recent group now comes out very slightly leaner. p = 0.89,
Hedges' g = 0.035, still negligible.

A result whose direction depends on whether the same player is counted once or
twice has no direction. That is the cleanest available statement of what this
data supports, and a stronger one than either p-value. The two periods are not
distinguishable on this metric, and the sign of the small gap between them is an
accident of who happened to have two good scoring seasons in a window rather
than one.

## What could have moved the number, and by how much

The ratio never changes within a player, so the only thing that can move a group
mean is which players are in the group. And since the ratio is close to a
position label, the most likely mechanism is a change in the position mix.

Two views settle it. What each group is made of, and how strongly the ratio
follows position in the first place.

In [9]:
cross_tab_heatmap(
    top20,
    col1="period",
    col2="position",
    normalize="row",
    show_counts=True,
    height=380,
    title="Position mix of the top 20, recent vs earlier",
)

The mix did move, and it moved the way the claim would want.

| | PG | SG | SF | PF | C |
| --- | ---: | ---: | ---: | ---: | ---: |
| recent | 35.0 % (14) | 17.5 % (7) | 10.0 % (4) | 27.5 % (11) | 10.0 % (4) |
| earlier | 30.0 % (12) | 22.5 % (9) | 10.0 % (4) | 20.0 % (8) | 17.5 % (7) |

Two swaps, and both of them run up the ratio scale. Two point guards (2.20)
replace two shooting guards (2.07), and three power forwards (1.89) replace
three centres (1.71). Small forward does not move. Arithmetically that should
have lifted the recent group's mean by about 0.02, and the next cell confirms it
did. The recent top 20 is a guard-heavier and centre-lighter set of leading
scorers, and if there is a story anywhere in this question, that is it.

It is a small story. Three centres out of twenty is one or two clubs changing
who takes their shots. The cells here hold between 4 and 14 players, which is
why the next chart carries a small-sample warning on every box.

In [10]:
grouped_box_plot(
    top20,
    group_col="position",
    value_col="height_to_weight",
    sort_by="median",
    min_n_flag=30,
    height=520,
    title="Height ÷ weight by position, all 80 player-seasons",
)

The five boxes step down almost perfectly in order: medians of 2.22 at point
guard, 2.09 at shooting guard, 2.06 at small forward, 1.91 at power forward and
1.68 at centre. The only pair that does not separate is shooting guard against
small forward, which are 0.008 apart on the mean and overlap almost entirely.

Every box carries a gold outline because every position has fewer than 30 rows.
That is honest labelling rather than a problem: the point of this chart is the
ordering, and the ordering is not in doubt. The single outlier dot under power
forward is Zion Williamson again.

So the ratio is close to a restatement of position, which is what makes the next
calculation the important one. If the position mix moved toward guards but the
mean did not move, something must have cancelled it.

In [11]:
table = top20.pivot_table(
    index="position",
    columns="period",
    values="height_to_weight",
    aggfunc=["count", "mean"],
    observed=True,
)
print(table.round(3).to_string(), end="\n\n")

# Re-weight the recent per-position means onto the earlier group's position
# shares. The two figures then differ only in ratio-within-position, so the
# residual is the part composition cannot explain.
mix = table[("count", EARLIER)] / table[("count", EARLIER)].sum()
matched = float((table[("mean", RECENT)] * mix).sum())
means = top20.groupby("period", observed=True)["height_to_weight"].mean()

print(f"observed gap, recent minus earlier    {means[RECENT] - means[EARLIER]:+.4f}")
print(f"gap holding the earlier position mix  {matched - means[EARLIER]:+.4f}")
print(f"the position mix on its own           {means[RECENT] - matched:+.4f}")

                              count                                                   mean                            
period   recent · 2022-23 & 2023-24 earlier · 2020-21 & 2021-22 recent · 2022-23 & 2023-24 earlier · 2020-21 & 2021-22
position                                                                                                              
PG                               14                          12                      2.169                       2.239
SG                                7                           9                      2.087                       2.060
SF                                4                           4                      2.015                       2.145
PF                               11                           8                      1.912                       1.862
C                                 4                           7                      1.692                       1.727

observed gap, recent minus earlier    -0.0034
g

Two effects, pointing opposite ways, both about the same size.

- The position mix on its own is worth **+0.0206**. More guards and fewer
  centres, exactly as the heat map showed.
- Holding the position mix fixed, the recent group is **−0.0240** below the
  earlier one. Within a position, the recent leading scorers carry slightly more
  weight for their height.
- The two nearly cancel and leave the observed **−0.0034**.

The per-position column shows where each part comes from, and it is not tidy:
the recent group is lower at point guard (2.169 against 2.239), small forward
(2.015 against 2.145) and centre (1.692 against 1.727), and higher at shooting
guard and power forward. Three down, two up, on cells holding between 4 and 14
players.

Neither component survives a serious look. Each is about a tenth of a standard
deviation, both are built on single-figure cell counts, and the −0.024 half is
largely one or two individuals moving in and out of a four-man position group.
The honest summary of this table is that the composition shift is real and
visible, its arithmetic effect on the metric is about +0.02, and the noise
around it is larger than the effect.

## Conclusion

**The claim is not supported. The top 20 scorers of 2022-23 and 2023-24 have
the same height-to-weight ratio as those of 2020-21 and 2021-22, and what small
difference exists points the other way.**

| | recent, 2022-23 & 2023-24 | earlier, 2020-21 & 2021-22 |
| --- | ---: | ---: |
| Player-seasons | 40 | 40 |
| Distinct players | 30 | 30 |
| Mean ratio | 2.0210 | 2.0244 |
| Median ratio | 1.985 | 1.975 |
| Standard deviation | 0.201 | 0.243 |
| Lowest / highest | 1.637 / 2.527 | 1.538 / 2.527 |

Welch's t-test on 40 against 40 gives t = −0.068, one-tailed p = 0.527,
two-tailed p = 0.946, and a mean difference of −0.0034 with a bootstrapped 95 %
interval from −0.105 to +0.090. Hedges' g is −0.015, negligible. H₀ stands.

The direction deserves its own sentence, because a one-tailed non-rejection is
the easiest place in statistics to hide something. The recent group came out
*lower*, not higher. Nothing turns on that, since the gap is one eighth of a
single rung of the measurement grid and the sign flips to positive as soon as
each player is counted once instead of twice, but a reader should not have to
work that out from a p-value of 0.527.

**What did change is the position mix, and it changed in the direction the claim
predicted.** Three centres left and three power forwards took their places; two
shooting guards left and two point guards took theirs. Since the ratio runs from
2.20 at point guard to 1.71 at centre, that shift alone was worth
+0.021 on the group mean, about a tenth of a standard deviation. A drift within
positions took −0.024 back off, and the two cancelled. So the sentence this data
supports is "the top 20 tilted toward guards", not "players became more agile".

Limits, in the order they matter:

- **The metric cannot measure what the claim is about.** Height and weight come
  from the bio page and never change: no player in this database has two
  different ratios. The number can only move through roster turnover in the top
  20. A league in which every player got leaner would show nothing here.
- **Height ÷ weight is not agility.** It is a build measurement. It cannot tell
  a quick guard from a slight one, and it says nothing about first step, lateral
  speed or leaping.
- **The groups are not independent.** 19 of 41 players are in both, supplying
  about 70 % of each side with the identical value. A two-sample t-test treats
  those as free information; they are the opposite of free, since they can only
  drag the estimate toward zero while shrinking the standard error.
- **n = 40 a side buys very little.** The interval rules out only shifts larger
  than about 0.1, a fifth of the guard-to-centre gap. A genuine change smaller
  than that was never detectable here.
- **"Top 20" is scoring volume.** It is the twenty highest scorers, which is all
  this data can rank on. The best twenty players is a different list.
- **Nothing here is causal**, and nothing here is about the league as a whole:
  40 player-seasons a side, from four seasons, describing the scoring leaders
  and nobody else.